In [7]:
import xarray as xr
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
import cmocean
import cartopy.crs as ccrs
import pandas as pd
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter, DayLocator
from matplotlib.ticker import FormatStrFormatter
import xroms
from xgcm import Grid
import cartopy.feature as cfeature
import cartopy
import numpy.ma as ma
import warnings
warnings.filterwarnings("ignore")#, category=FutureWarning)

In [2]:
path = '/mnt/syno/ROMS_1by111/output_roms_1by_111/river/output/history/roms_river_bob_1km_hist_'
file_extension = '.nc'

# Create a list of file paths for all 365 files
file_paths = [f'{path}{day:04d}{file_extension}' for day in range(213, 244)]
ds_1 = xr.open_mfdataset(file_paths,compat='override', combine='by_coords',data_vars='minimal', coords='minimal')#, parallel=True,chunks={"ocean_time": 1})


# path = '/home/shrikant/Mayur_Gachake/test/roms_river_bob_1km_hist_0162.nc'
# ds_1 = xr.open_mfdataset(path)
ds_1 = ds_1.drop_vars('Vtransform')
ds_1 = ds_1.assign(Vtransform=2)
ds_1, xgrid = xroms.roms_dataset(ds_1,include_cell_volume=True, Vtransform=2, include_Z0=True)
ds_1.xroms.set_grid(xgrid)

ds_1 = ds_1.chunk({'ocean_time':1,"s_rho":-1})
ml  = ds_1.xroms.mld(thresh=0.2)

i_min= 300; i_max=1500; j_min=800 ; j_max = 2000 # i:lon points, j:lat points, i_max = 3070, j_max = 2443
temp = ds_1.temp.chunk({'ocean_time': 1,"s_rho": -1}).isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))
salt = ds_1.salt.chunk({'ocean_time': 1,"s_rho": -1}).isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))
vvel = ds_1.v_northward.chunk({'ocean_time': 1,"s_rho": -1}).isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))
uvel = ds_1.u_eastward.chunk({'ocean_time': 1,"s_rho": -1}).isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))
wvel = ds_1.w.chunk({'ocean_time': 1,"s_w": -1}).isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))


depths = [0,-2,-4,-6,-8,-10,-12,-14,-16,-18,-20,-22,-24,-26,-28,-30,-32,-34,-36,-38,-40,-45,-50,-55,-60,-65,-70,\
    -75,-80,-85,-90,-95,-100,-110,-120,-130,-140,-150,-160,-170,-180,-190,-200,-300,-400,-500]


data_t = temp
depth_t = temp['z_rho']
data_temp = xgrid.transform(data_t, 'Z', np.array(depths),
                                        target_data=depth_t,
                                        method='linear')
data_s = salt
depth_s = salt['z_rho']
data_salt = xgrid.transform(data_s, 'Z', np.array(depths),
                                        target_data=depth_s,
                                        method='linear')
data_v = vvel
depth_v = vvel['z_rho']
data_vvel = xgrid.transform(data_v, 'Z', np.array(depths),
                                        target_data=depth_v,
                                        method='linear')

data_u = uvel
depth_u = uvel['z_rho']
data_uvel = xgrid.transform(data_u, 'Z', np.array(depths),
                                        target_data=depth_u,
                                        method='linear')

data_w = wvel
depth_w = wvel['z_w']
data_wvel = xgrid.transform(data_w, 'Z', np.array(depths),
                                        target_data=depth_w,
                                        method='linear')
data_wvel = data_wvel.rename({'z_w':'z_rho'})

i_min= 300; i_max=1500; j_min=800 ; j_max = 2000
rho0 = ds_1.rho.chunk({'ocean_time': 1,"s_rho": -1}).isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))

# rho0 = ds_1.rho.chunk({'ocean_time': 1,"s_rho": -1})

depths = [0,-2,-4,-6,-8,-10,-12,-14,-16,-18,-20,-22,-24,-26,-28,-30,-32,-34,-36,-38,-40,-45,-50,-55,-60,-65,-70,\
    -75,-80,-85,-90,-95,-100,-110,-120,-130,-140,-150,-160,-170,-180,-190,-200,-300,-400,-500]

data_r = rho0
depth_r = rho0['z_rho']
data_rho = xgrid.transform(data_r, 'Z', np.array(depths),
                                        target_data=depth_r,
                                        method='linear')

def create(var,data):
    attrs = ds_1[var].attrs
    vlat,vlon = 'lat_rho', 'lon_rho'
    lat, lon = data[vlat][:,0].values, data[vlon][0,:].values
    times = pd.to_datetime(data['ocean_time'].values)   #####np.abs(depths)],\
    da = xr.DataArray(data, coords = [times, lat, lon, depths],
                        dims = ['time', 'lat', 'lon', 'depth'])
    da = da.sortby('time').transpose(...,'lat', 'lon')
    da.attrs = attrs
    da['lat'].attrs = {'standard_name':'Latitude', 'units':'degrees_north'}
    da['lon'].attrs = {'standard_name':'Longitude', 'units':'degrees_east'}
    da['depth'].attrs = {'standard_name':'Depth', 'units': 'meter'}
    return da
t1 = create('temp',data_temp) ; t2 = t1[:,1:,:,:]
s1 = create('salt',data_salt) ; s2 = s1[:,1:,:,:]
v1 = create('v_northward',data_vvel)
u1 = create('u_eastward',data_uvel)
w1 = create('w',data_wvel)
r1 = create('rho',data_rho); r2 = r1[:,1:,:,:]
# # # above variables have dimensions: [time, depth, lat, lon]

lat_min = 4; lat_max=14
lon_min=79; lon_max = 91
t1 = t1.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
s1 = s1.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
u1 = u1.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
v1 = v1.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
w1 = w1.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
r1 = r1.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))


attrs = ml.attrs
mld1 = ml.isel(eta_rho=slice(j_min,j_max),xi_rho = slice(i_min,i_max))
vlat,vlon = 'lat_rho', 'lon_rho'
lat, lon = mld1[vlat][:,0].values, mld1[vlon][0,:].values
times = pd.to_datetime(mld1['ocean_time'].values)
dmld = xr.DataArray(mld1, coords = [times, lat, lon],\
                        dims = ['time', 'lat', 'lon'])
dmld = dmld.sortby('time').transpose(...,'lat', 'lon')
dmld.attrs = attrs
dmld['lat'].attrs = {'standard_name':'Latitude', 'units':'degrees_north'}
dmld['lon'].attrs = {'standard_name':'Longitude', 'units':'degrees_east'}
mld = dmld * -1

In [33]:
# mld= mld.sel(lon=slice(lon_min,lon_max),lat=slice(lat_min,lat_max))
# mld

In [5]:
# from netCDF4 import Dataset
# import numpy as np

# # ncfile = '00_ROMS_variables_5Jun_30Jun_2015.nc'
# dsss = xr.Dataset({
#     "mld" : mld
# })

# # Save all variables to a single NetCDF file
# dsss.to_netcdf("2_MLD_ROMS_Aug_2015.nc")

In [29]:
# for i in range(0,10):
#     fig, ax = plt.subplots(figsize=(8,6))
#     mld[i,:,:].plot(cmap=cmocean.cm.deep,levels=np.arange(-60.01, -4.99, 5))
#     ax.set_title(f"{i+1} August 2015, lon : 85$^o$E",fontsize=14)
#     ax.set_xlabel("Longitude (°E)",fontsize=14)
#     ax.set_ylabel("Latitude (°N)",fontsize=14)
#     plt.tight_layout()
#     plt.savefig(f'{i+1}_August_MLD_ROMS_2015',bbox_inches='tight')

In [32]:
# mld_spatial_mean_Aug = mld.mean(dim=["lat", "lon"])

# mld_spatial_mean_Aug.plot(marker="o")
# plt.ylabel("Mean MLD (m)")
# plt.title("Spatial mean of MLD")
# plt.show()